# GeoIncite Employee Business Card Generator

Generates AES-encrypted `.vcf.enc` files compatible with `eBusinessCard.html` (uses CryptoJS's default OpenSSL-style "Salted__" AES-256-CBC scheme).

**Workflow:**
1. Run the setup cell below once
2. Fill in an employee's details in the "Single card" section and run it
3. Upload the generated `.vcf.enc` file to `cards/data/` in your repo
4. Share the printed URL with the employee

For adding many employees at once, see the **Batch generation** section at the bottom.

## Setup
Run this once per session.

In [ ]:
# Install dependency if needed (uncomment the line below if not already installed)
# %pip install pycryptodome --quiet

import base64
import hashlib
import os
import secrets
import string
from pathlib import Path

from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad

print("Setup OK.")

In [ ]:
def evp_bytes_to_key(password: bytes, salt: bytes, key_len=32, iv_len=16):
    """Replicates OpenSSL/CryptoJS's default MD5-based key derivation."""
    d = d_i = b""
    while len(d) < key_len + iv_len:
        d_i = hashlib.md5(d_i + password + salt).digest()
        d += d_i
    return d[:key_len], d[key_len:key_len + iv_len]


def encrypt_vcard(vcard_text: str, passphrase: str) -> str:
    salt = os.urandom(8)
    key, iv = evp_bytes_to_key(passphrase.encode("utf-8"), salt)
    cipher = AES.new(key, AES.MODE_CBC, iv)
    padded = pad(vcard_text.encode("utf-8"), AES.block_size)
    ciphertext = cipher.encrypt(padded)
    blob = b"Salted__" + salt + ciphertext
    return base64.b64encode(blob).decode("ascii")


def decrypt_vcard(encrypted_b64: str, passphrase: str) -> str:
    """Round-trip check: decrypts the same way the browser's CryptoJS does."""
    raw = base64.b64decode(encrypted_b64)
    salt, ciphertext = raw[8:16], raw[16:]
    key, iv = evp_bytes_to_key(passphrase.encode("utf-8"), salt)
    cipher = AES.new(key, AES.MODE_CBC, iv)
    return unpad(cipher.decrypt(ciphertext), AES.block_size).decode("utf-8")


def build_vcard(first_name, last_name, title, email, phone, org="GeoIncite",
                 url="https://geoincite.github.io", credentials="", prefix="",
                 street="", city="", state="", zip_code="", country=""):
    """
    first_name/last_name: kept separate so prefixes like "Dr." don't get
        misparsed as part of the last name.
    prefix: honorific, e.g. "Dr." -> vCard N property, 4th component.
    credentials: post-nominal letters, e.g. "PE, PhD" -> vCard N property,
        5th component (suffix).
    street/city/state/zip_code/country: mapped to the vCard ADR property.
        Leave any blank to omit that component.
    """
    full_name = " ".join(p for p in [prefix, first_name, last_name] if p)
    display_name = f"{full_name}, {credentials}" if credentials else full_name

    lines = [
        "BEGIN:VCARD",
        "VERSION:3.0",
        f"FN:{display_name}",
        f"N:{last_name};{first_name};;{prefix};{credentials}",
        f"ORG:{org}",
        f"TITLE:{title}",
        f"EMAIL:{email}",
        f"TEL;TYPE=CELL:{phone}",
    ]

    if any([street, city, state, zip_code, country]):
        lines.append(f"ADR;TYPE=WORK:;;{street};{city};{state};{zip_code};{country}")

    lines.append(f"URL:{url}")
    lines.append("END:VCARD")
    lines.append("")
    return "\n".join(lines)


def generate_strong_passphrase(length=20):
    alphabet = string.ascii_letters + string.digits
    return "".join(secrets.choice(alphabet) for _ in range(length))


def make_card(emp_id, first_name, last_name, title, email, phone,
              prefix="", credentials="", street="", city="", state="",
              zip_code="", country="", passphrase=None, out_dir="."):
    """
    Builds, encrypts, saves, and prints the share URL for one employee card.
    If passphrase is None, a strong random one is generated automatically.
    Returns (file_path, passphrase, url).
    """
    if passphrase is None:
        passphrase = generate_strong_passphrase()

    vcard = build_vcard(first_name, last_name, title, email, phone,
                         prefix=prefix, credentials=credentials,
                         street=street, city=city, state=state,
                         zip_code=zip_code, country=country)
    encrypted = encrypt_vcard(vcard, passphrase)

    out_path = Path(out_dir) / f"{emp_id}.vcf.enc"
    out_path.write_text(encrypted)

    # Round-trip sanity check before declaring success
    check = decrypt_vcard(encrypted, passphrase)
    assert "BEGIN:VCARD" in check, "Round-trip verification failed!"

    url = f"https://geoincite.com/cards/eBusinessCard.html?id={emp_id}#{passphrase}"

    print(f"File saved: {out_path}")
    print(f"Passphrase: {passphrase}")
    print(f"URL:        {url}")
    print("Round-trip verified OK.")

    return out_path, passphrase, url

print("Functions loaded.")

## Single card

Fill in one employee's details, then run the cell. It builds the vCard, encrypts it, saves `{id}.vcf.enc` in the same folder as this notebook, and prints the URL to send them.

Leave `credentials`, `prefix`, or any address field as `""` to omit them.

In [ ]:
file_path, passphrase, url = make_card(
    emp_id="emp3",
    prefix="Dr.",
    first_name="Marcus",
    last_name="Reid",
    credentials="PE, PhD",
    title="Principal LiDAR Scientist",
    email="marcus.reid@geoincite.com",
    phone="+1-720-555-0192",
    street="1200 Survey Way",
    city="Denver",
    state="CO",
    zip_code="80202",
    country="USA",
    # passphrase="MyOwnPassphrase123",  # uncomment to set your own instead of a random one
)

## Batch generation

For adding many employees at once, fill in a list of dicts (or load from CSV) and run each through `make_card`.

**CSV option:** if you have a spreadsheet with columns like `emp_id, prefix, first_name, last_name, credentials, title, email, phone, street, city, state, zip_code, country`, load it with `pandas.read_csv(...)` and pass each row's fields to `make_card(**row.to_dict())`.

In [ ]:
employees = [
    dict(emp_id="emp4", first_name="Priya", last_name="Nair",
         title="Field Survey Lead", email="priya.nair@geoincite.com",
         phone="+1-720-555-0177"),
    dict(emp_id="emp5", first_name="Sam", last_name="Chen",
         title="GIS Analyst", email="sam.chen@geoincite.com",
         phone="+1-720-555-0163", credentials="GISP"),
]

results = []
for emp in employees:
    print(f"--- {emp['emp_id']} ---")
    results.append(make_card(**emp))
    print()

print(f"Generated {len(results)} cards.")

## Notes

- **Keep the passphrase list somewhere private** (password manager, private spreadsheet) — it's the only thing protecting each card's data, and it's not recoverable from the `.enc` file alone.
- **Don't commit this notebook's output cells** if they contain real employee passphrases — clear outputs before committing (`Cell → All Output → Clear`), or keep this notebook outside your public repo entirely (e.g. in a separate private `tools/` location).
- To **update** an existing employee, just re-run `make_card` with the same `emp_id` and new details — it overwrites the local file. Re-upload it to `cards/data/` to replace the live one. The share URL only changes if you also change the passphrase.